# Part 2: NeRF 3D Reconstruction

This notebook orchestrates Part 2 training and rendering using helper modules:
- `rendering.py`
- `nerf_model.py`
- `train_part2.py`
- `part2_utils.py`

Start with a smoke run, then scale to full training.

In [1]:
from pathlib import Path
import numpy as np
import torch

In [2]:
from part2_utils import ensure_dir, plot_training_curves, save_depth_png, save_rgb_png, set_seed
from train_part2 import build_part2_data, render_test_trajectory, train_nerf_part2

In [7]:
# Core configuration (mid run profile)
CFG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "data_path": "lego_200x200.npz",
    "output_dir": "images/output/part2",
    "hidden_dim": 256,
    "n_layers": 8,
    "pos_freqs": 10,
    "dir_freqs": 4,
    "n_coarse": 32,
    "n_fine": 32,
    "batch_rays": 1024,
    "n_steps": 120,
    "eval_every": 30,
    "chunk_size": 4096,
    "lr": 5e-4,
    "near_override": 2.0,
    "far_override": 6.0,
}

set_seed(CFG["seed"])
ensure_dir(CFG["output_dir"])
CFG

{'seed': 42,
 'device': 'cpu',
 'data_path': 'lego_200x200.npz',
 'output_dir': 'images/output/part2',
 'hidden_dim': 256,
 'n_layers': 8,
 'pos_freqs': 10,
 'dir_freqs': 4,
 'n_coarse': 32,
 'n_fine': 32,
 'batch_rays': 1024,
 'n_steps': 120,
 'eval_every': 30,
 'chunk_size': 4096,
 'lr': 0.0005,
 'near_override': 2.0,
 'far_override': 6.0}

In [8]:
# Quick data sanity check (no training)
data = build_part2_data(CFG["data_path"], device=CFG["device"])
print("K shape:", tuple(data["k"].shape))
print("Train rays:", len(data["train_dataset"]))
print("Val images:", tuple(data["val_images"].shape))
print("Test poses:", tuple(data["test_c2ws"].shape))

K shape: (3, 3)
Train rays: 4000000
Val images: (10, 200, 200, 3)
Test poses: (60, 4, 4)


In [9]:
# Smoke training run (increase n_steps and batch_rays later)
results = train_nerf_part2(
    data_path=CFG["data_path"],
    output_dir=CFG["output_dir"],
    device=CFG["device"],
    seed=CFG["seed"],
    hidden_dim=CFG["hidden_dim"],
    n_layers=CFG["n_layers"],
    pos_freqs=CFG["pos_freqs"],
    dir_freqs=CFG["dir_freqs"],
    n_coarse=CFG["n_coarse"],
    n_fine=CFG["n_fine"],
    n_steps=CFG["n_steps"],
    batch_rays=CFG["batch_rays"],
    lr=CFG["lr"],
    eval_every=CFG["eval_every"],
    chunk_size=CFG["chunk_size"],
    near_override=CFG["near_override"],
    far_override=CFG["far_override"],
)
results["metrics"]

[train_nerf_part2] start device=cpu steps=120 batch_rays=1024 coarse=32 fine=32 near=2.000 far=6.000 eval_every=30
[train_nerf_part2] step 1/120 loss=0.198607 coarse=0.079104 fine=0.190696 iter=4.738s rays_per_sec=216
[train_nerf_part2] step 6/120 loss=0.177449 coarse=0.099374 fine=0.167511 iter=3.472s rays_per_sec=295
[train_nerf_part2] step 12/120 loss=0.118951 coarse=0.204241 fine=0.098527 iter=3.675s rays_per_sec=279
[train_nerf_part2] step 18/120 loss=0.099429 coarse=0.165862 fine=0.082843 iter=3.678s rays_per_sec=278
[train_nerf_part2] step 24/120 loss=0.089733 coarse=0.068595 fine=0.082873 iter=3.418s rays_per_sec=300
[train_nerf_part2] step 30/120 loss=0.073142 coarse=0.082201 fine=0.064922 iter=3.778s rays_per_sec=271
[train_nerf_part2] eval step 30/120 val_psnr=11.669dB best=11.669dB (new best, saved checkpoint) eval=57.92s
[train_nerf_part2] step 36/120 loss=0.075291 coarse=0.086622 fine=0.066629 iter=3.665s rays_per_sec=279
[train_nerf_part2] step 42/120 loss=0.067799 coars

{'best_psnr': 15.872197151184082,
 'best_step': 120,
 'final_loss': 0.03776073828339577,
 'near': 2.0,
 'far': 6.0,
 'n_steps': 120,
 'batch_rays': 1024,
 'n_coarse': 32,
 'n_fine': 32,
 'val_psnr_hist': [11.668710708618164,
  12.311869859695435,
  13.410923480987549,
  15.872197151184082],
 'eval_steps': [30, 60, 90, 120],
 'total_seconds': 691.2401971999789,
 'avg_seconds_per_step': 5.76033497666649,
 'log_every': 6}

In [10]:
# Save training curves
curve_dir = Path(CFG["output_dir"]) / "curves"
plot_training_curves(
    loss_hist=results["loss_hist"],
    eval_steps=results["eval_steps"],
    val_psnr_hist=results["val_psnr_hist"],
    output_dir=curve_dir,
)
print("Saved curves to", curve_dir)

Saved curves to images\output\part2\curves


In [ ]:
# Render 60 test views to NPY (RGB + depth)
models = results["models"]
data = results["data"]
image_hw = (data["val_images"].shape[1], data["val_images"].shape[2])

render_test_trajectory(
    model_coarse=models["coarse"],
    model_fine=models["fine"],
    k=data["k"],
    test_c2ws=data["test_c2ws"],
    image_hw=image_hw,
    output_dir=CFG["output_dir"],
    near=results["metrics"]["near"],
    far=results["metrics"]["far"],
    n_coarse=CFG["n_coarse"],
    n_fine=CFG["n_fine"],
    chunk_size=CFG["chunk_size"],
    device=CFG["device"],
)
print("Saved test trajectory npy files.")

In [ ]:
# Optional: convert NPY outputs to PNG files
rgb_npy_dir = Path(CFG["output_dir"]) / "test_rgb_npy"
depth_npy_dir = Path(CFG["output_dir"]) / "test_depth_npy"
rgb_png_dir = Path(CFG["output_dir"]) / "test_renders"
depth_png_dir = Path(CFG["output_dir"]) / "depth_maps"

rgb_png_dir.mkdir(parents=True, exist_ok=True)
depth_png_dir.mkdir(parents=True, exist_ok=True)

for npy_path in sorted(rgb_npy_dir.glob("view_*.npy")):
    img = np.load(npy_path)
    save_rgb_png(img, rgb_png_dir / f"{npy_path.stem}.png")

for npy_path in sorted(depth_npy_dir.glob("view_*.npy")):
    dep = np.load(npy_path)
    save_depth_png(dep, depth_png_dir / f"{npy_path.stem}.png")

print("Saved PNG outputs:", rgb_png_dir, depth_png_dir)